# 07 — Iterative Backtest Notebook
Standalone experiment notebook. It rebuilds the same Silver/Gold feature
engineering as Pipe-1 from the raw source tables, trains the same two-stage
LightGBM model as `03_train_model.py`, and evaluates validation/internal
test/final inference in **iterative** mode:
- the scored horizon never sees its true `quantite`;
- week `t+1` uses the model prediction from week `t` to rebuild `lag_1`,
rolling stats, trends, zero-rates, expanding stats, etc.;
- validation and internal test are therefore block-forecast backtests.


%pip install lightgbm==4.3.0
dbutils.library.restartPython()


In [ ]:
import math
import sys
sys.path.append("./")

import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from mlflow.models import infer_signature
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType
from pyspark.sql.window import Window



# -----------------------------------------------------------------------------
# Autonomous configuration: raw tables, splits, features, model params
# -----------------------------------------------------------------------------
NOM_EQUIPE = "telecacaton"
TABLE_PREDICTIONS = f"workspace.default.predictions_equipe_{NOM_EQUIPE}"

TBL_TRAIN = "workspace.default.histo_ventes_train"
TBL_TEST = "workspace.default.histo_ventes_test"
TBL_AGENCE = "workspace.default.donnees_agence"
TBL_ARTICLES = "workspace.default.donnees_articles"
TBL_FACTURATION = "workspace.default.donnees_facturation"

PAIR_KEYS = ["code_agence", "code_article"]

TRAIN_END_WEEK_ID = 202426
VAL_START_WEEK_ID = 202427
VAL_END_WEEK_ID = 202452
INTERNAL_TEST_START_WEEK_ID = 202501
INTERNAL_TEST_END_WEEK_ID = 202526
FINAL_INFERENCE_START_WEEK_ID = 202527
FINAL_INFERENCE_END_WEEK_ID = 202552

SEED = 42
OUTLIER_PERCENTILE = 0.995
ANOMALY_MULTIPLIER = 10.0
ANOMALY_ROLL_WINDOW = 26

LAGS_ALL = [1, 2, 4, 8, 13, 26, 52, 104]
ROLLING_WINDOWS = [4, 8, 13, 26, 52]
ROLLING_MEDIAN_WINDOWS = [4, 13]

FEATURES_NUMERIC = [
    "lag_1", "lag_2", "lag_4", "lag_8", "lag_13", "lag_26", "lag_52", "lag_104",
    "roll_mean_4", "roll_mean_8", "roll_mean_13", "roll_mean_26", "roll_mean_52",
    "roll_std_4", "roll_std_8", "roll_std_13", "roll_std_26", "roll_std_52",
    "roll_median_4", "roll_median_13",
    "zero_rate_26", "zero_rate_52", "pair_zero_rate_expanding",
    "trend_8", "ratio_n1_vs_mean", "yoy_ratio",
    "pair_mean", "pair_median", "pair_max", "pair_count", "pair_cv",
    "sem_mean", "sem_max", "sem_median",
    "agence_mean", "agence_median",
    "article_mean", "article_median",
    "n_active_weeks",
    "fac_prix_unit", "fac_pct_pro", "fac_nb_chantiers", "fac_nb_achats",
    "annee", "num_sem", "sin_sem", "cos_sem",
    "is_summer_trough", "is_xmas_trough",
]
FEATURES_CATEGORICAL = [
    "art_specialite_enc",
    "art_famille_enc",
    "art_marque_enc",
    "art_mdd_enc",
    "ag_region_enc",
]
FEATURES = FEATURES_NUMERIC + FEATURES_CATEGORICAL

LGB_PARAMS_ZERO = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_child_samples": 50,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "scale_pos_weight": 1.5,
    "n_jobs": -1,
    "seed": SEED,
    "verbose": -1,
}
LGB_NUM_ROUNDS_ZERO = 3000
LGB_EARLY_STOP_ZERO = 75

LGB_PARAMS_QTY = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "None",
    "learning_rate": 0.03,
    "num_leaves": 255,
    "min_child_samples": 30,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "max_bin": 511,
    "n_jobs": -1,
    "seed": SEED,
    "verbose": -1,
}
LGB_NUM_ROUNDS_QTY = 5000
LGB_EARLY_STOP_QTY = 100

ZERO_THRESHOLD_GRID = [0.30, 0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
MLFLOW_EXPERIMENT = f"/Shared/sgdb2026_{NOM_EQUIPE}_iterative_notebook"


# -----------------------------------------------------------------------------
# Autonomous utility functions
# -----------------------------------------------------------------------------
EPS = 1e-10


def wape_numpy(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + EPS))


def wape_lgb_feval(y_pred, dataset):
    y_true = dataset.get_label()
    return "wape", wape_numpy(y_true, y_pred), False


def apply_non_iterative_feature_fallbacks(df, features):
    """Residual safety fallback for lags/rolling stats that are still null."""
    out = df.copy()
    feature_set = set(features)
    if "pair_mean" in out.columns:
        for c in feature_set:
            if not c.startswith("lag_"):
                continue
            try:
                n = int(c.split("_")[1])
            except (IndexError, ValueError):
                continue
            if n < 26 and c in out.columns:
                out[c] = out[c].fillna(out["pair_mean"])

    for short, long in [
        ("roll_mean_4", "roll_mean_26"),
        ("roll_mean_8", "roll_mean_26"),
        ("roll_mean_13", "roll_mean_26"),
        ("roll_std_4", "roll_std_26"),
        ("roll_std_8", "roll_std_26"),
        ("roll_std_13", "roll_std_26"),
        ("roll_median_4", "roll_median_13"),
    ]:
        if short in out.columns and long in out.columns:
            out[short] = out[short].fillna(out[long])
    return out

mlflow.set_experiment(MLFLOW_EXPERIMENT)


## 0. Controls


In [ ]:
# Exact iterative threshold tuning reruns the whole 26-week recursive validation
# loop for every threshold. It is the cleanest protocol, but it is expensive.
RUN_EXACT_ITERATIVE_THRESHOLD_SWEEP = True

# Use a smaller grid while debugging if needed, for example [0.55, 0.65, 0.75].
ITERATIVE_THRESHOLD_GRID = ZERO_THRESHOLD_GRID

OUT_VAL_ITER = "workspace.default.iterative_nb_val_predictions"
OUT_INTERNAL_TEST_ITER = "workspace.default.iterative_nb_internal_test_predictions"
OUT_FINAL_ITER = f"workspace.default.iterative_nb_predictions_equipe_{NOM_EQUIPE}"

print(f"Train           : <= {TRAIN_END_WEEK_ID}")
print(f"Validation      : {VAL_START_WEEK_ID}..{VAL_END_WEEK_ID}")
print(f"Internal test   : {INTERNAL_TEST_START_WEEK_ID}..{INTERNAL_TEST_END_WEEK_ID}")
print(f"Final inference : {FINAL_INFERENCE_START_WEEK_ID}..{FINAL_INFERENCE_END_WEEK_ID}")
print(f"Iter threshold grid: {ITERATIVE_THRESHOLD_GRID}")


## 1. Pipe-1 Silver Reconstruction


In [ ]:
def _add_time_columns(df, semaine_col="semaine"):
    return (
        df.withColumn("annee", F.split(F.col(semaine_col), "-").getItem(0).cast("int"))
          .withColumn("num_sem", F.split(F.col(semaine_col), "-").getItem(1).cast("int"))
          .withColumn("week_id", F.col("annee") * F.lit(100) + F.col("num_sem"))
    )


def _encode_column(df, src, dst):
    if src not in df.columns:
        return df.withColumn(dst, F.lit(-1).cast("int"))
    dim = (
        df.select(src)
        .distinct()
        .withColumn(dst, F.dense_rank().over(Window.orderBy(src)) - F.lit(1))
    )
    return df.join(dim, src, "left")


def build_silver_ventes(train_raw):
    raw = (
        train_raw
        .transform(_add_time_columns)
        .withColumnRenamed("quantite", "quantite_raw")
    )

    pair_stats = (
        raw.groupBy(*PAIR_KEYS)
        .agg(
            F.sum("quantite_raw").alias("_pair_sum"),
            F.expr("percentile_approx(quantite_raw, 0.995)").alias("_pair_p995"),
            F.expr("percentile_approx(quantite_raw, 0.5)").alias("_pair_median"),
        )
        .withColumn("is_dead_pair", (F.col("_pair_sum") == 0).cast("tinyint"))
    )

    capped = (
        raw.join(pair_stats, PAIR_KEYS, "left")
        .withColumn(
            "_cap_value",
            F.when(F.col("is_dead_pair") == 1, F.lit(None))
             .otherwise(F.greatest(F.col("_pair_p995"), F.col("_pair_median") * F.lit(2.0))),
        )
        .withColumn(
            "is_capped",
            (
                F.col("_cap_value").isNotNull()
                & (F.col("quantite_raw") > F.col("_cap_value"))
            ).cast("tinyint"),
        )
        .withColumn(
            "quantite_capped",
            F.when(F.col("is_capped") == 1, F.col("_cap_value").cast("double"))
             .otherwise(F.col("quantite_raw").cast("double")),
        )
    )

    roll_w = (
        Window.partitionBy(*PAIR_KEYS)
        .orderBy("week_id")
        .rowsBetween(-ANOMALY_ROLL_WINDOW, -1)
    )
    with_roll = (
        capped
        .withColumn("_roll_median", F.expr("percentile_approx(quantite_capped, 0.5)").over(roll_w))
        .withColumn(
            "is_anomaly",
            (
                F.col("_roll_median").isNotNull()
                & (F.col("_roll_median") > F.lit(0))
                & (F.col("quantite_capped") > F.lit(ANOMALY_MULTIPLIER) * F.col("_roll_median"))
            ).cast("tinyint"),
        )
        .withColumn(
            "quantite",
            F.when(F.col("is_anomaly") == 1, F.col("_roll_median"))
             .otherwise(F.col("quantite_capped"))
             .cast("long"),
        )
    )

    smooth_w = (
        Window.partitionBy(*PAIR_KEYS)
        .orderBy("week_id")
        .rowsBetween(-13, -1)
    )

    return (
        with_roll
        .withColumn("quantite_smooth", F.avg("quantite").over(smooth_w))
        .select(
            "semaine", "annee", "num_sem", "week_id",
            "code_agence", "code_article",
            F.col("quantite_raw").cast("long").alias("quantite_raw"),
            F.col("quantite").cast("long").alias("quantite"),
            F.col("quantite_smooth").cast("double").alias("quantite_smooth"),
            F.col("is_anomaly").cast("tinyint"),
            F.col("is_capped").cast("tinyint"),
            F.col("is_dead_pair").cast("tinyint"),
        )
    )


def build_articles_encoded(articles_raw):
    df = articles_raw
    mapping = [
        ("specialite", "art_specialite_enc"),
        ("famille", "art_famille_enc"),
        ("marque", "art_marque_enc"),
        ("article_mdd", "art_mdd_enc"),
    ]
    for src, dst in mapping:
        df = _encode_column(df, src, dst)

    keep = ["code_agence", "code_article"] + [dst for _, dst in mapping]
    return df.select(*keep).dropDuplicates(["code_agence", "code_article"])


def build_agences_encoded(agences_raw):
    src = "region" if "region" in agences_raw.columns else "ag_region"
    df = agences_raw.withColumnRenamed(src, "ag_region")
    df = _encode_column(df, "ag_region", "ag_region_enc")
    return df.select("code_agence", "ag_region_enc").dropDuplicates(["code_agence"])


def build_facturation_lagged(fac_raw):
    def _pick(*candidates, default=None):
        for c in candidates:
            if c in fac_raw.columns:
                return F.col(c)
        return F.lit(default)

    year_col = _pick("annee", "year")
    month_col = _pick("mois", "month")

    monthly = (
        fac_raw
        .withColumn("_annee", year_col.cast("int"))
        .withColumn("_mois", month_col.cast("int"))
        .groupBy("code_agence", "code_article", "_annee", "_mois")
        .agg(
            F.sum(_pick("sum_montant", default=0.0)).alias("_sum_montant"),
            F.sum(_pick("sum_quantite", default=0.0)).alias("_sum_quantite"),
            F.sum(_pick("nb_achats", default=0.0)).alias("fac_nb_achats"),
            F.sum(_pick("nb_achats_par_professionnels", default=0.0)).alias("_nb_pro"),
            F.sum(_pick("nb_chantiers", default=0.0)).alias("fac_nb_chantiers"),
        )
        .withColumn(
            "fac_prix_unit",
            F.when((F.col("_sum_quantite").isNull()) | (F.col("_sum_quantite") == 0), None)
             .otherwise(F.col("_sum_montant") / F.col("_sum_quantite")),
        )
        .withColumn(
            "fac_pct_pro",
            F.when((F.col("fac_nb_achats").isNull()) | (F.col("fac_nb_achats") == 0), None)
             .otherwise(F.col("_nb_pro") / F.col("fac_nb_achats")),
        )
    )

    return (
        monthly
        .withColumn("_shifted_mois", F.col("_mois") + F.lit(2))
        .withColumn(
            "_join_annee",
            F.when(F.col("_shifted_mois") > F.lit(12), F.col("_annee") + F.lit(1))
             .otherwise(F.col("_annee")),
        )
        .withColumn(
            "_join_mois",
            F.when(F.col("_shifted_mois") > F.lit(12), F.col("_shifted_mois") - F.lit(12))
             .otherwise(F.col("_shifted_mois")),
        )
        .select(
            "code_agence", "code_article", "_join_annee", "_join_mois",
            "fac_prix_unit", "fac_pct_pro",
            F.col("fac_nb_chantiers").cast("double"),
            F.col("fac_nb_achats").cast("double"),
        )
    )


def build_silver_panel(cleaned, test_raw):
    test = (
        test_raw
        .transform(_add_time_columns)
        .withColumn("quantite_raw", F.lit(None).cast("long"))
        .withColumn("quantite", F.lit(None).cast("long"))
        .withColumn("quantite_smooth", F.lit(None).cast("double"))
        .withColumn("is_anomaly", F.lit(0).cast("tinyint"))
        .withColumn("is_capped", F.lit(0).cast("tinyint"))
        .withColumn("is_dead_pair", F.lit(0).cast("tinyint"))
        .select(*cleaned.columns)
    )

    pair_w = Window.partitionBy(*PAIR_KEYS)
    return cleaned.unionByName(test).withColumn(
        "is_dead_pair",
        F.max("is_dead_pair").over(pair_w).cast("tinyint"),
    )


In [ ]:
train_raw = spark.table(TBL_TRAIN)
test_raw = spark.table(TBL_TEST)
agences_raw = spark.table(TBL_AGENCE)
articles_raw = spark.table(TBL_ARTICLES)
fac_raw = spark.table(TBL_FACTURATION)

silver_ventes = build_silver_ventes(train_raw).cache()
articles_enc = build_articles_encoded(articles_raw).cache()
agences_enc = build_agences_encoded(agences_raw).cache()
fac_lagged = build_facturation_lagged(fac_raw).cache()
panel = build_silver_panel(silver_ventes, test_raw).cache()

print(f"Silver panel rows: {panel.count():,}")


## 2. Pipe-1 Gold Feature Builder


In [ ]:
def build_features_from_y(panel_with_y, articles_enc, agences_enc, fac):
    df = panel_with_y
    pair_order = Window.partitionBy(*PAIR_KEYS).orderBy("week_id")

    for n in LAGS_ALL:
        df = df.withColumn(f"lag_{n}", F.lag("y", n).over(pair_order))

    def _lookback(n):
        return (
            Window.partitionBy(*PAIR_KEYS)
            .orderBy("week_id")
            .rowsBetween(-n, -1)
        )

    for n in ROLLING_WINDOWS:
        w = _lookback(n)
        df = (
            df.withColumn(f"roll_mean_{n}", F.avg("y").over(w))
              .withColumn(f"roll_std_{n}", F.stddev("y").over(w))
        )
    for n in ROLLING_MEDIAN_WINDOWS:
        df = df.withColumn(
            f"roll_median_{n}",
            F.expr("percentile_approx(y, 0.5)").over(_lookback(n)),
        )

    df = df.withColumn(
        "_y_is_zero",
        F.when(F.col("y").isNull(), F.lit(None).cast("double"))
         .when(F.col("y") == 0, F.lit(1.0))
         .otherwise(F.lit(0.0)),
    )
    df = (
        df
        .withColumn("zero_rate_26", F.avg("_y_is_zero").over(_lookback(26)))
        .withColumn("zero_rate_52", F.avg("_y_is_zero").over(_lookback(52)))
        .withColumn(
            "pair_zero_rate_expanding",
            F.avg("_y_is_zero").over(
                Window.partitionBy(*PAIR_KEYS)
                .orderBy("week_id")
                .rowsBetween(Window.unboundedPreceding, -1)
            ),
        )
    )

    recent_w = Window.partitionBy(*PAIR_KEYS).orderBy("week_id").rowsBetween(-4, -1)
    prev_w = Window.partitionBy(*PAIR_KEYS).orderBy("week_id").rowsBetween(-8, -5)
    df = (
        df
        .withColumn("_mean_recent4", F.avg("y").over(recent_w))
        .withColumn("_mean_prev4", F.avg("y").over(prev_w))
        .withColumn(
            "trend_8",
            F.when(F.col("_mean_prev4").isNull(), F.lit(None).cast("double"))
             .otherwise(
                 F.least(
                     F.greatest(
                         (F.col("_mean_recent4") - F.col("_mean_prev4"))
                         / (F.col("_mean_prev4") + F.lit(1.0)),
                         F.lit(-5.0),
                     ),
                     F.lit(5.0),
                 )
             ),
        )
        .withColumn(
            "yoy_ratio",
            F.when(
                F.col("lag_104").isNull() | (F.col("lag_104") == 0),
                F.lit(None).cast("double"),
            ).otherwise(F.col("lag_52") / F.col("lag_104")),
        )
    )

    pair_exp = (
        Window.partitionBy(*PAIR_KEYS)
        .orderBy("week_id")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    df = (
        df
        .withColumn("pair_mean", F.avg("y").over(pair_exp))
        .withColumn("pair_median", F.expr("percentile_approx(y, 0.5)").over(pair_exp))
        .withColumn("pair_max", F.max("y").over(pair_exp))
        .withColumn("pair_count", F.count("y").over(pair_exp))
        .withColumn("_pair_std", F.stddev("y").over(pair_exp))
        .withColumn(
            "pair_cv",
            F.when((F.col("pair_mean").isNull()) | (F.col("pair_mean") == 0), None)
             .otherwise(F.col("_pair_std") / (F.col("pair_mean") + F.lit(1e-6))),
        )
        .withColumn(
            "ratio_n1_vs_mean",
            F.when((F.col("pair_mean").isNull()) | (F.col("pair_mean") == 0), None)
             .otherwise(F.col("lag_52") / F.col("pair_mean")),
        )
        .withColumn("n_active_weeks", F.sum((F.col("y") > 0).cast("double")).over(pair_exp))
    )

    season_w = (
        Window.partitionBy(*PAIR_KEYS, "num_sem")
        .orderBy("annee")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    df = (
        df
        .withColumn("sem_mean", F.avg("y").over(season_w))
        .withColumn("sem_max", F.max("y").over(season_w))
        .withColumn("sem_median", F.expr("percentile_approx(y, 0.5)").over(season_w))
    )

    ag_w = Window.partitionBy("code_agence").orderBy("week_id").rowsBetween(Window.unboundedPreceding, -1)
    art_w = Window.partitionBy("code_article").orderBy("week_id").rowsBetween(Window.unboundedPreceding, -1)
    df = (
        df
        .withColumn("agence_mean", F.avg("y").over(ag_w))
        .withColumn("agence_median", F.expr("percentile_approx(y, 0.5)").over(ag_w))
        .withColumn("article_mean", F.avg("y").over(art_w))
        .withColumn("article_median", F.expr("percentile_approx(y, 0.5)").over(art_w))
    )

    two_pi = F.lit(2 * math.pi)
    df = (
        df
        .withColumn("sin_sem", F.sin(two_pi * F.col("num_sem") / F.lit(52.0)))
        .withColumn("cos_sem", F.cos(two_pi * F.col("num_sem") / F.lit(52.0)))
        .withColumn("is_summer_trough", ((F.col("num_sem") >= 30) & (F.col("num_sem") <= 35)).cast("tinyint"))
        .withColumn("is_xmas_trough", ((F.col("num_sem") >= 50) | (F.col("num_sem") == 1)).cast("tinyint"))
    )

    df = df.join(articles_enc, PAIR_KEYS, "left").join(agences_enc, "code_agence", "left")

    df = (
        df
        .withColumn(
            "_join_mois",
            F.least(F.lit(12), F.greatest(F.lit(1), F.ceil(F.col("num_sem") / F.lit(4.333)))),
        )
        .withColumn("_join_annee", F.col("annee"))
        .join(fac, ["code_agence", "code_article", "_join_annee", "_join_mois"], "left")
        .drop("_join_annee", "_join_mois")
    )

    base_cols = [
        "semaine", "week_id", "code_agence", "code_article",
        "quantite", "quantite_raw", "quantite_smooth",
        "is_anomaly", "is_capped", "is_dead_pair",
    ]
    for c in FEATURES:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast("double"))

    return df.select(*base_cols, *FEATURES)


def panel_with_history_y(panel, history_end_week_id, pred_sdf=None):
    if pred_sdf is not None:
        base = panel.join(pred_sdf, ["semaine", "code_agence", "code_article"], "left")
    else:
        base = panel.withColumn("_iter_pred", F.lit(None).cast("double"))

    return base.withColumn(
        "y",
        F.when(F.col("_iter_pred").isNotNull(), F.col("_iter_pred"))
         .when(F.col("week_id") <= F.lit(history_end_week_id), F.col("quantite").cast("double"))
         .otherwise(F.lit(None).cast("double")),
    )


def build_static_features(history_end_week_id, start_week_id, end_week_id):
    return (
        build_features_from_y(
            panel_with_history_y(panel, history_end_week_id),
            articles_enc,
            agences_enc,
            fac_lagged,
        )
        .filter((F.col("week_id") >= F.lit(start_week_id)) & (F.col("week_id") <= F.lit(end_week_id)))
    )


## 3. Static Train/Validation Features for LightGBM Early Stopping
LightGBM's native early stopping expects a fixed validation matrix. The true
recursive validation matrix depends on the model's own predictions, so this
notebook uses a leakage-free one-shot validation matrix for early stopping,
then tunes/evaluates WAPE with the iterative scorer below.


In [ ]:
train_features_sdf = build_static_features(TRAIN_END_WEEK_ID, 0, TRAIN_END_WEEK_ID).cache()
val_static_sdf = build_static_features(TRAIN_END_WEEK_ID, VAL_START_WEEK_ID, VAL_END_WEEK_ID).cache()

cols_needed = ["semaine", "code_agence", "code_article", "week_id", "quantite", "is_dead_pair"] + FEATURES
train_pd = train_features_sdf.select(*cols_needed).toPandas()
val_static_pd = val_static_sdf.select(*cols_needed).toPandas()
val_static_pd[FEATURES] = apply_non_iterative_feature_fallbacks(val_static_pd[FEATURES], FEATURES)

print(f"Train rows: {len(train_pd):,}")
print(f"Static validation rows for early stopping: {len(val_static_pd):,}")


## 4. Same Two-Stage LightGBM Model as `03_train_model.py`


In [ ]:
def build_xy(df: pd.DataFrame):
    X = df[FEATURES].copy()
    for c in FEATURES_CATEGORICAL:
        if c in X.columns:
            X[c] = X[c].astype("category")
    y = df["quantite"].astype(float).values
    is_zero = (y == 0).astype(int)
    return X, y, is_zero


X_tr, y_tr, z_tr = build_xy(train_pd)
X_va_static, y_va_static, z_va_static = build_xy(val_static_pd)

print(f"Zero rate train: {z_tr.mean():.3f}   static val: {z_va_static.mean():.3f}")


In [ ]:
RUN_ID = None
with mlflow.start_run(run_name="iterative_backtest_train") as run:
    RUN_ID = run.info.run_id
    mlflow.log_params({
        "n_features": len(FEATURES),
        "train_rows": len(train_pd),
        "static_val_rows": len(val_static_pd),
        "train_end": TRAIN_END_WEEK_ID,
        "val_start": VAL_START_WEEK_ID,
        "val_end": VAL_END_WEEK_ID,
        "internal_test_start": INTERNAL_TEST_START_WEEK_ID,
        "internal_test_end": INTERNAL_TEST_END_WEEK_ID,
        "iterative_threshold_sweep": RUN_EXACT_ITERATIVE_THRESHOLD_SWEEP,
    })

    dtrain_z = lgb.Dataset(X_tr, label=z_tr, categorical_feature=FEATURES_CATEGORICAL)
    dval_z = lgb.Dataset(X_va_static, label=z_va_static, reference=dtrain_z, categorical_feature=FEATURES_CATEGORICAL)
    model_zero = lgb.train(
        LGB_PARAMS_ZERO,
        dtrain_z,
        num_boost_round=LGB_NUM_ROUNDS_ZERO,
        valid_sets=[dtrain_z, dval_z],
        valid_names=["train", "static_val"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=LGB_EARLY_STOP_ZERO, first_metric_only=True),
            lgb.log_evaluation(period=100),
        ],
    )

    nz = y_tr > 0
    X_tr_nz = X_tr.loc[nz].reset_index(drop=True)
    y_tr_nz = y_tr[nz]
    nz_va = y_va_static > 0
    X_va_nz = X_va_static.loc[nz_va].reset_index(drop=True)
    y_va_nz = y_va_static[nz_va]

    qty_params = dict(LGB_PARAMS_QTY)
    qty_params["objective"] = "regression_l1"
    qty_params["metric"] = "None"

    dtrain_q = lgb.Dataset(X_tr_nz, label=y_tr_nz, categorical_feature=FEATURES_CATEGORICAL)
    dval_q = lgb.Dataset(X_va_nz, label=y_va_nz, reference=dtrain_q, categorical_feature=FEATURES_CATEGORICAL)
    model_qty = lgb.train(
        qty_params,
        dtrain_q,
        num_boost_round=LGB_NUM_ROUNDS_QTY,
        valid_sets=[dtrain_q, dval_q],
        valid_names=["train", "static_val"],
        feval=wape_lgb_feval,
        callbacks=[
            lgb.early_stopping(stopping_rounds=LGB_EARLY_STOP_QTY, first_metric_only=True),
            lgb.log_evaluation(period=100),
        ],
    )

    mlflow.log_param("best_iteration_zero", model_zero.best_iteration)
    mlflow.log_param("best_iteration_qty", model_qty.best_iteration)

    X_sig = X_tr.head(5).copy()
    for c in FEATURES_CATEGORICAL:
        if c in X_sig.columns:
            X_sig[c] = X_sig[c].astype(int)
    mlflow.lightgbm.log_model(
        model_zero,
        artifact_path="iter_zero_classifier",
        signature=infer_signature(X_sig, model_zero.predict(X_tr.head(5))),
        input_example=X_sig.head(1),
    )

    X_q_sig = X_tr_nz.head(5).copy()
    for c in FEATURES_CATEGORICAL:
        if c in X_q_sig.columns:
            X_q_sig[c] = X_q_sig[c].astype(int)
    mlflow.lightgbm.log_model(
        model_qty,
        artifact_path="iter_qty_regressor",
        signature=infer_signature(X_q_sig, model_qty.predict(X_tr_nz.head(5))),
        input_example=X_q_sig.head(1),
    )


## 5. Iterative Scorer


In [ ]:
pred_schema = StructType([
    StructField("semaine", StringType(), False),
    StructField("code_agence", LongType(), False),
    StructField("code_article", LongType(), False),
    StructField("_iter_pred", DoubleType(), True),
])


def _empty_pred_sdf():
    return spark.createDataFrame([], schema=pred_schema)


def _prediction_rows_to_sdf(rows):
    if len(rows) == 0:
        return _empty_pred_sdf()
    return spark.createDataFrame(
        pd.DataFrame(rows, columns=["semaine", "code_agence", "code_article", "_iter_pred"]),
        schema=pred_schema,
    )


def score_horizon_iterative(history_end_week_id, start_week_id, end_week_id, threshold, label):
    pred_rows = []
    out_parts = []

    weeks = [row["week_id"] for row in (
        panel
        .filter((F.col("week_id") >= F.lit(start_week_id)) & (F.col("week_id") <= F.lit(end_week_id)))
        .select("week_id")
        .distinct()
        .orderBy("week_id")
        .collect()
    )]

    for week_id in weeks:
        print(f"[{label}] scoring week_id={week_id} with {len(pred_rows):,} previous predictions")
        pred_sdf = _prediction_rows_to_sdf(pred_rows)
        feature_week_sdf = (
            build_features_from_y(
                panel_with_history_y(panel, history_end_week_id, pred_sdf),
                articles_enc,
                agences_enc,
                fac_lagged,
            )
            .filter(F.col("week_id") == F.lit(week_id))
            .select(*cols_needed)
        )

        week_pd = feature_week_sdf.toPandas()
        week_pd[FEATURES] = apply_non_iterative_feature_fallbacks(week_pd[FEATURES], FEATURES)

        X_week = week_pd[FEATURES].copy()
        for c in FEATURES_CATEGORICAL:
            if c in X_week.columns:
                X_week[c] = X_week[c].astype("category")

        p_zero = model_zero.predict(X_week, num_iteration=model_zero.best_iteration)
        qty = np.clip(model_qty.predict(X_week, num_iteration=model_qty.best_iteration), 0.0, None)
        pred = np.where(p_zero > threshold, 0.0, qty)
        pred = np.where(week_pd["is_dead_pair"].values == 1, 0.0, pred)

        week_out = week_pd[["semaine", "code_agence", "code_article", "week_id", "quantite", "is_dead_pair"]].copy()
        week_out["p_zero"] = p_zero
        week_out["qty_pred"] = qty
        week_out["prediction"] = pred
        week_out["prediction_int"] = np.clip(np.round(pred), 0, None).astype(np.int64)
        week_out["split"] = label
        out_parts.append(week_out)

        append_rows = week_out[["semaine", "code_agence", "code_article", "prediction"]].copy()
        append_rows = append_rows.rename(columns={"prediction": "_iter_pred"})
        pred_rows.extend(append_rows.itertuples(index=False, name=None))

    return pd.concat(out_parts, ignore_index=True)


def score_horizon_static_once(history_end_week_id, start_week_id, end_week_id, threshold, label):
    sdf = build_static_features(history_end_week_id, start_week_id, end_week_id).select(*cols_needed)
    df = sdf.toPandas()
    df[FEATURES] = apply_non_iterative_feature_fallbacks(df[FEATURES], FEATURES)
    X = df[FEATURES].copy()
    for c in FEATURES_CATEGORICAL:
        if c in X.columns:
            X[c] = X[c].astype("category")
    p_zero = model_zero.predict(X, num_iteration=model_zero.best_iteration)
    qty = np.clip(model_qty.predict(X, num_iteration=model_qty.best_iteration), 0.0, None)
    pred = np.where(p_zero > threshold, 0.0, qty)
    pred = np.where(df["is_dead_pair"].values == 1, 0.0, pred)
    out = df[["semaine", "code_agence", "code_article", "week_id", "quantite", "is_dead_pair"]].copy()
    out["p_zero"] = p_zero
    out["qty_pred"] = qty
    out["prediction"] = pred
    out["prediction_int"] = np.clip(np.round(pred), 0, None).astype(np.int64)
    out["split"] = label
    return out


## 6. Threshold Tuning on Iterative Validation


In [ ]:
threshold_rows = []

if RUN_EXACT_ITERATIVE_THRESHOLD_SWEEP:
    for thr in ITERATIVE_THRESHOLD_GRID:
        val_pred_thr = score_horizon_iterative(
            history_end_week_id=TRAIN_END_WEEK_ID,
            start_week_id=VAL_START_WEEK_ID,
            end_week_id=VAL_END_WEEK_ID,
            threshold=thr,
            label=f"validation_thr_{thr}",
        )
        wape = wape_numpy(val_pred_thr["quantite"].values, val_pred_thr["prediction"].values)
        threshold_rows.append({"threshold": thr, "iterative_val_wape": wape})
        print(f"threshold={thr} -> iterative validation WAPE={wape:.5f}")
else:
    for thr in ITERATIVE_THRESHOLD_GRID:
        val_pred_thr = score_horizon_static_once(
            history_end_week_id=TRAIN_END_WEEK_ID,
            start_week_id=VAL_START_WEEK_ID,
            end_week_id=VAL_END_WEEK_ID,
            threshold=thr,
            label=f"validation_static_thr_{thr}",
        )
        wape = wape_numpy(val_pred_thr["quantite"].values, val_pred_thr["prediction"].values)
        threshold_rows.append({"threshold": thr, "iterative_val_wape": wape})
        print(f"threshold={thr} -> static masked validation WAPE={wape:.5f}")

threshold_df = pd.DataFrame(threshold_rows).sort_values("iterative_val_wape")
display(threshold_df)

best_threshold = float(threshold_df.iloc[0]["threshold"])
best_val_wape = float(threshold_df.iloc[0]["iterative_val_wape"])
print(f"Best threshold: {best_threshold} -> validation WAPE={best_val_wape:.5f}")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_param("iter_best_zero_threshold", best_threshold)
    mlflow.log_metric("iter_val_wape", best_val_wape)


## 7. Final Iterative Validation and Internal Test


In [ ]:
val_iter = score_horizon_iterative(
    history_end_week_id=TRAIN_END_WEEK_ID,
    start_week_id=VAL_START_WEEK_ID,
    end_week_id=VAL_END_WEEK_ID,
    threshold=best_threshold,
    label="validation",
)
val_iter_wape = wape_numpy(val_iter["quantite"].values, val_iter["prediction"].values)
print(f"Validation iterative WAPE: {val_iter_wape:.5f}")

internal_test_iter = score_horizon_iterative(
    history_end_week_id=VAL_END_WEEK_ID,
    start_week_id=INTERNAL_TEST_START_WEEK_ID,
    end_week_id=INTERNAL_TEST_END_WEEK_ID,
    threshold=best_threshold,
    label="internal_test",
)
internal_test_wape = wape_numpy(internal_test_iter["quantite"].values, internal_test_iter["prediction"].values)
print(f"Internal test iterative WAPE: {internal_test_wape:.5f}")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric("iter_val_wape_final", val_iter_wape)
    mlflow.log_metric("iter_internal_test_wape", internal_test_wape)


In [ ]:
def write_predictions_pd(df, table_name, include_quantite=True):
    cols = ["semaine", "code_agence", "code_article"]
    if include_quantite:
        cols.append("quantite")
    cols += ["p_zero", "qty_pred", "prediction", "prediction_int"]

    out_sdf = (
        spark.createDataFrame(df[cols])
        .withColumn("code_agence", F.col("code_agence").cast(LongType()))
        .withColumn("code_article", F.col("code_article").cast(LongType()))
        .withColumn("prediction_int", F.col("prediction_int").cast(LongType()))
    )
    (
        out_sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Wrote {out_sdf.count():,} rows to {table_name}")


write_predictions_pd(val_iter, OUT_VAL_ITER, include_quantite=True)
write_predictions_pd(internal_test_iter, OUT_INTERNAL_TEST_ITER, include_quantite=True)


## 8. Final Leaderboard-Horizon Iterative Inference


In [ ]:
final_iter = score_horizon_iterative(
    history_end_week_id=INTERNAL_TEST_END_WEEK_ID,
    start_week_id=FINAL_INFERENCE_START_WEEK_ID,
    end_week_id=FINAL_INFERENCE_END_WEEK_ID,
    threshold=best_threshold,
    label="final_inference",
)

submission = final_iter[["semaine", "code_agence", "code_article", "prediction_int"]].copy()
submission = submission.rename(columns={"prediction_int": "quantite"})
submission["quantite"] = np.clip(submission["quantite"].round(), 0, None).astype(np.int64)

submission_sdf = (
    spark.createDataFrame(submission)
    .withColumn("code_agence", F.col("code_agence").cast(LongType()))
    .withColumn("code_article", F.col("code_article").cast(LongType()))
    .withColumn("quantite", F.col("quantite").cast(LongType()))
)

(
    submission_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_FINAL_ITER)
)

print(f"Wrote iterative final submission candidate: {OUT_FINAL_ITER}")
print(f"Rows: {submission_sdf.count():,}")
print(f"Positive predictions: {submission_sdf.filter(F.col('quantite') > 0).count():,}")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_param("iterative_final_table", OUT_FINAL_ITER)
    mlflow.log_metric("iterative_final_rows", submission_sdf.count())
